# 02 - Train a LoRA baseline on macOS MPS

This notebook launches the repository's response-only-loss trainer with conservative settings for an Apple Silicon Mac with 64 GB unified memory.

The first run is intentionally a smoke test. It uses rank 8, sequence length 256, one epoch, and only `q_proj` and `v_proj`. Increase capacity only after a complete save-and-reload cycle succeeds.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import torch


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Repository root not found")


ROOT = find_repo_root(Path.cwd().resolve())
ADAPTER_NAME = "my_adapter"
BASE_MODEL = "ibm-granite/granite-4.1-3b"
DATA_DIR = ROOT / "workspaces" / ADAPTER_NAME
OUTPUT_DIR = ROOT / "outputs" / "my-library" / ADAPTER_NAME / "granite-4.1-3b" / "lora"
RUN_TRAINING = False

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this Mac training notebook")
print("MPS ready; output will be:", OUTPUT_DIR)

## Confirm inputs

Run notebook 01 first or point `DATA_DIR` at an existing validated dataset.

In [ ]:
required = [DATA_DIR / "train.jsonl", DATA_DIR / "validation.jsonl", DATA_DIR / "io.yaml"]
missing = [path for path in required if not path.is_file()]
if missing:
    print("Missing inputs:", *missing, sep="\n- ")
else:
    print("All training inputs exist")

## Build the smoke-test command

The Hugging Face Trainer detects MPS on Apple Silicon. The current recipe keeps model precision at its safe default and uses gradient checkpointing to reduce activation memory.

In [ ]:
command = [
    sys.executable,
    str(ROOT / "scripts" / "train_adapter.py"),
    "--technology",
    "lora",
    "--base-model",
    BASE_MODEL,
    "--train-file",
    str(DATA_DIR / "train.jsonl"),
    "--validation-file",
    str(DATA_DIR / "validation.jsonl"),
    "--output-dir",
    str(OUTPUT_DIR),
    "--rank",
    "8",
    "--alpha",
    "16",
    "--target-modules",
    "q_proj,v_proj",
    "--epochs",
    "1",
    "--batch-size",
    "1",
    "--gradient-accumulation-steps",
    "8",
    "--max-length",
    "256",
    "--gradient-checkpointing",
]
print(" ".join(map(str, command)))

## Train

Set `RUN_TRAINING = True` only after checking the paths and command. The first execution downloads the model and may take time.

In [ ]:
if RUN_TRAINING:
    if missing:
        raise FileNotFoundError(f"Missing required inputs: {missing}")
    subprocess.run(command, cwd=ROOT, check=True)
    torch.mps.synchronize()
    torch.mps.empty_cache()
else:
    print("Training disabled. Review the command, then set RUN_TRAINING = True.")

## Validate the saved adapter

A successful training run must produce PEFT config and safetensors weights. Copy the approved `io.yaml` beside them before validation.

In [ ]:
if RUN_TRAINING:
    io_text = (DATA_DIR / "io.yaml").read_text(encoding="utf-8")
    (OUTPUT_DIR / "io.yaml").write_text(io_text, encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "granite_adapter_guide", "validate-adapter", str(OUTPUT_DIR)],
        cwd=ROOT,
        check=True,
    )

## Next experiment

After the smoke test works, increase one dimension at a time: dataset size, sequence length, epochs, target modules, then rank. Keep an untouched test set and compare quality rather than training loss alone.